# ZooKeeper: duplicate detection

For each issue that was later marked as a duplicate, rank all the earlier ZooKeeper issues by text similarity and check how high the real original comes. Compares TF-IDF against sentence embeddings (MiniLM), with a random ranking as the baseline.

In [ ]:
!pip install jira sentence-transformers

In [2]:
import re
import numpy as np
import pandas as pd
from jira import JIRA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

jira = JIRA('https://issues.apache.org/jira')

In [3]:
issues = jira.search_issues('project=ZOOKEEPER', maxResults=False,
                            fields='summary,description,created,issuelinks')
print(f"Found {len(issues)} issues")

rows = []
pairs = set()

for issue in issues:
    num = int(issue.key.split('-')[1])
    rows.append({
        'key': issue.key,
        'num': num,
        'summary': issue.fields.summary or '',
        'description': issue.fields.description or '',
        'created': issue.fields.created
    })

    for link in issue.fields.issuelinks or []:
        if 'duplicate' not in link.type.name.lower():
            continue
        other = getattr(link, 'inwardIssue', None) or getattr(link, 'outwardIssue', None)
        if other is None or not other.key.startswith('ZOOKEEPER-'):
            continue
        other_num = int(other.key.split('-')[1])
        if other_num == num:
            continue
        pairs.add((max(num, other_num), min(num, other_num)))

df = pd.DataFrame(rows).sort_values('num').reset_index(drop=True)
df.to_csv('zookeeper_issues.csv', index=False)

known = set(df['num'])
pairs = [(new, old) for new, old in pairs if new in known and old in known]
pairs_df = pd.DataFrame(sorted(pairs), columns=['newer', 'older'])
pairs_df.to_csv('zookeeper_duplicate_pairs.csv', index=False)

print(f"Duplicate pairs: {len(pairs_df)}")
print(f"Issues that are a duplicate of an earlier one: {pairs_df['newer'].nunique()}")

Found 4935 issues
Duplicate pairs: 192
Issues that are a duplicate of an earlier one: 179


In [4]:
def clean_text(text):
    text = re.sub(r'\{code.*?\}.*?\{code\}', ' ', text, flags=re.DOTALL)
    text = re.sub(r'\{noformat\}.*?\{noformat\}', ' ', text, flags=re.DOTALL)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = (df['summary'] + '. ' + df['description']).apply(clean_text)

nums = df['num'].values
num_to_idx = {n: i for i, n in enumerate(nums)}

targets = pairs_df.groupby('newer')['older'].apply(set).to_dict()
query_nums = sorted(targets)
query_idx = [num_to_idx[n] for n in query_nums]

print(f"Queries: {len(query_nums)}")
print(f"Average number of earlier issues to search through: {np.mean(query_idx):.0f}")

Queries: 179
Average number of earlier issues to search through: 2781


In [5]:
def evaluate(sims, name):
    ranks = []
    for row, q in enumerate(query_nums):
        scores = sims[row].copy()
        scores[nums >= q] = -np.inf
        order = np.argsort(-scores)
        target_idx = {num_to_idx[t] for t in targets[q]}
        rank = next(r for r, i in enumerate(order, start=1) if i in target_idx)
        ranks.append(rank)
    ranks = np.array(ranks)
    return {
        'method': name,
        'recall@1': round(float((ranks <= 1).mean()), 3),
        'recall@5': round(float((ranks <= 5).mean()), 3),
        'recall@10': round(float((ranks <= 10).mean()), 3),
        'MRR': round(float((1 / ranks).mean()), 3),
        'median rank': float(np.median(ranks))
    }, ranks

results = []

In [6]:
rng = np.random.default_rng(42)
random_sims = rng.random((len(query_nums), len(df)))
res, _ = evaluate(random_sims, 'Random')
results.append(res)
print(res)

{'method': 'Random', 'recall@1': 0.0, 'recall@5': 0.0, 'recall@10': 0.0, 'MRR': 0.002, 'median rank': 1186.0}


In [7]:
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words='english', sublinear_tf=True)
X = vectorizer.fit_transform(df['text'])
print(f"TF-IDF feature shape: {X.shape}")

tfidf_sims = linear_kernel(X[query_idx], X)
res, tfidf_ranks = evaluate(tfidf_sims, 'TF-IDF')
results.append(res)
print(res)

TF-IDF feature shape: (4935, 20000)
{'method': 'TF-IDF', 'recall@1': 0.374, 'recall@5': 0.581, 'recall@10': 0.687, 'MRR': 0.473, 'median rank': 3.0}


In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
emb = model.encode(df['text'].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True)

minilm_sims = emb[query_idx] @ emb.T
res, minilm_ranks = evaluate(minilm_sims, 'MiniLM')
results.append(res)
print(res)

{'method': 'MiniLM', 'recall@1': 0.358, 'recall@5': 0.564, 'recall@10': 0.665, 'MRR': 0.451, 'median rank': 4.0}


In [9]:
results_df = pd.DataFrame(results)
results_df.to_csv('duplicate_detection_results.csv', index=False)
print(results_df.round(3).to_string(index=False))

method  recall@1  recall@5  recall@10   MRR  median rank
Random     0.000     0.000      0.000 0.002       1186.0
TF-IDF     0.374     0.581      0.687 0.473          3.0
MiniLM     0.358     0.564      0.665 0.451          4.0


In [10]:
summaries = df['summary'].values

for row in range(0, len(query_nums), max(1, len(query_nums) // 5)):
    q = query_nums[row]
    scores = tfidf_sims[row].copy()
    scores[nums >= q] = -np.inf
    top = np.argsort(-scores)[:3]

    print(f"ZOOKEEPER-{q}: {summaries[num_to_idx[q]]}")
    for t in sorted(targets[q]):
        print(f"   real original   ZOOKEEPER-{t}: {summaries[num_to_idx[t]]}")
    for i in top:
        hit = 'yes' if nums[i] in targets[q] else 'no '
        print(f"   guess ({hit}) {scores[i]:.2f}  ZOOKEEPER-{nums[i]}: {summaries[i]}")
    print(f"   rank of real original: {tfidf_ranks[row]}")
    print()

ZOOKEEPER-61: Address (remove) use of sleep(#) in client/server test cases.
   real original   ZOOKEEPER-54: remove sleeps in the tests.
   guess (no ) 0.11  ZOOKEEPER-28: Incompatible client and server list detection
   guess (yes) 0.09  ZOOKEEPER-54: remove sleeps in the tests.
   guess (no ) 0.06  ZOOKEEPER-60: Get cppunit tests running as part of Hudson CI
   rank of real original: 2

ZOOKEEPER-1694: ZooKeeper Leader sends a repeated NEWLEADER quorum packet to followers
   real original   ZOOKEEPER-1324: Remove Duplicate NEWLEADER packets from the Leader to the Follower.
   guess (no ) 0.26  ZOOKEEPER-1498: Zab1.0 sends NEWLEADER packet twice
   guess (yes) 0.18  ZOOKEEPER-1324: Remove Duplicate NEWLEADER packets from the Leader to the Follower.
   guess (no ) 0.16  ZOOKEEPER-1194: Two possible race conditions during leader establishment
   rank of real original: 2

ZOOKEEPER-2318: segfault in auth_completion_func
   real original   ZOOKEEPER-1485: client xid overflow is not handle